In [2]:
# Корень репозитория (где pyproject.toml) → в sys.path добавляется src/
import sys
from pathlib import Path

for _p in [Path.cwd(), *Path.cwd().parents]:
    _src = _p / "src"
    if (_p / "pyproject.toml").is_file() and _src.is_dir():
        _s = str(_src)
        if _s not in sys.path:
            sys.path.insert(0, _s)
        break

from models.cnn_encoder import RawCNNClassifier
from metrics.metrics import compute_classification_metrics
import torch

In [3]:
batch_size = 32
channels = 6
time_steps = 100
num_classes = 4

In [4]:
model = RawCNNClassifier(
    in_channels=channels,
    num_classes=num_classes,
    d_model=128,
    hidden_channels=(64, 128, 128),
    kernel_size=5,
    encoder_dropout=0.1,
    head_dropout=0.2,
)

x_raw = torch.randn(batch_size, channels, time_steps)

logits = model(x_raw)

print(logits.shape)

torch.Size([32, 4])


In [5]:
@torch.no_grad()
def evaluate_model(
    model,
    dataloader,
    device,
    class_names=None,
):
    model.eval()

    all_preds = []
    all_targets = []
    total_loss = 0.0
    total_samples = 0

    criterion = nn.CrossEntropyLoss()

    for batch in dataloader:
        x_raw, y = batch

        x_raw = x_raw.to(device)
        y = y.to(device)

        logits = model(x_raw)
        loss = criterion(logits, y)

        preds = torch.argmax(logits, dim=1)

        batch_size = y.size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size

        all_preds.append(preds.cpu().numpy())
        all_targets.append(y.cpu().numpy())

    y_pred = np.concatenate(all_preds)
    y_true = np.concatenate(all_targets)

    metrics = compute_classification_metrics(
        y_true=y_true,
        y_pred=y_pred,
        class_names=class_names,
    )

    metrics["loss"] = total_loss / total_samples

    return metrics

In [6]:
metrics = evaluate_model(
    model=model,
    dataloader=test_loader,
    device=device,
    class_names=["walking", "resting", "running", "badminton"],
)

print("Loss:", metrics["loss"])
print("Accuracy:", metrics["accuracy"])
print("Macro F1:", metrics["macro_f1"])
print("Weighted F1:", metrics["weighted_f1"])

print("Confusion matrix:")
print(metrics["confusion_matrix"])

print("Classification report:")
print(metrics["classification_report"])

NameError: name 'test_loader' is not defined

In [ ]:
experiment_result = {
    "experiment_name": "E1_raw_cnn_baseline",
    "modalities": ["raw"],
    "fusion": None,
    "encoder": "1D CNN",
    "loss": metrics["loss"],
    "accuracy": metrics["accuracy"],
    "macro_f1": metrics["macro_f1"],
    "weighted_f1": metrics["weighted_f1"],
    "confusion_matrix": metrics["confusion_matrix"],
}